# Canal de carros: teste realista (SDXL)
Modelo RealVisXL V4.0 (licença openrail++, permite uso comercial) + VAE fp16-fix. 5 cenas x 2 variações em 16:9.
Requer GPU T4 e Internet On.

In [ ]:
# libera memória de execuções anteriores nesta sessão
import gc, torch
for _n in ['pipe']:
    if _n in globals(): del globals()[_n]
gc.collect(); torch.cuda.empty_cache()
!pip install -q -U diffusers transformers accelerate safetensors

In [ ]:
import os, shutil, time
from PIL import Image
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, DPMSolverMultistepScheduler

OUT = '/kaggle/working/honda_real'
shutil.rmtree(OUT, ignore_errors=True); os.makedirs(OUT, exist_ok=True)
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = StableDiffusionXLPipeline.from_pretrained('SG161222/RealVisXL_V4.0', vae=vae, torch_dtype=torch.float16)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, algorithm_type='dpmsolver++', final_sigmas_type='sigma_min', use_karras_sigmas=True)
pipe = pipe.to('cuda')
print('ok', torch.cuda.memory_allocated() / 1e9, 'GB')

In [ ]:
ESTILO = 'automotive photography, professional photo, sharp focus, natural lighting, highly detailed, 35mm lens'
NEGATIVO = ('logo, emblem, badge text, watermark, text, letters, cartoon, illustration, 3d render, cgi, toy, '
            'deformed wheels, extra wheels, warped body, blurry, low quality, people')

cenas = [
    'small red 1975 Honda Civic hatchback parked on a quiet Tokyo street, golden hour',
    'white 1988 Honda CR-X sport coupe driving on a winding mountain road, motion blur background',
    'red 1991 Honda NSX sports car in a dark studio, dramatic rim lighting, three quarter front view',
    'championship white Honda Civic Type R hatchback with large rear wing on a race track, low angle',
    'silver Honda S2000 roadster with top down on a coastal highway at sunset',
]
W, H = 1024, 576
SEEDS = [42, 7]
grid = []
t0 = time.time()
for i, c in enumerate(cenas, 1):
    row = []
    for v, sd in enumerate(SEEDS):
        g = torch.Generator('cuda').manual_seed(sd)
        img = pipe(prompt=f'{c}, {ESTILO}', negative_prompt=NEGATIVO, width=W, height=H,
                   num_inference_steps=25, guidance_scale=6, generator=g).images[0]
        img.save(f'{OUT}/{i:03d}_{chr(97+v)}.png'); row.append(img)
    grid.append(row); print(f'cena {i:03d} ok  ({time.time()-t0:.0f}s)')

In [ ]:
tw, th = 512, 288
contato = Image.new('RGB', (tw * 2, th * len(grid)), 'white')
for r, row in enumerate(grid):
    for v, im in enumerate(row):
        contato.paste(im.resize((tw, th)), (v * tw, r * th))
contato.save('/kaggle/working/folha_contato_honda_real.png')
shutil.make_archive('/kaggle/working/honda_real', 'zip', OUT)
contato